# Week 02 Challenge Solution

Tasks:
1. Download the Titanic dataset via the Kaggle API
2. Identify data types
3. Transform `Sex` into a two-column (0/1) matrix
4. Create a subset: survived AND (female > 45 OR male < 20)
5. Count how many passengers were selected

## Download the Titanic dataset via Kaggle API

In [1]:
import os
import shutil
import warnings

warnings.filterwarnings("ignore")

# Point Kaggle to the credentials file in the repo root
os.environ['KAGGLE_CONFIG_DIR'] = '/workspace'

import kaggle

dataset = 'yasserh/titanic-dataset'
file_name = 'Titanic-Dataset.csv'
download_path = '/workspace/Week_02/challenge/data'

os.makedirs(download_path, exist_ok=True)

kaggle.api.authenticate()
kaggle.api.dataset_download_file(
    dataset,
    file_name=file_name,
    path=download_path,
    force=True
)

# Unzip if downloaded as .zip
zip_path = os.path.join(download_path, file_name + '.zip')
if os.path.exists(zip_path):
    import zipfile
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(download_path)
    os.remove(zip_path)

csv_path = os.path.join(download_path, file_name)
print(f'Dataset saved to: {csv_path}')

Dataset URL: https://www.kaggle.com/datasets/yasserh/titanic-dataset
Dataset saved to: /workspace/Week_02/challenge/data/Titanic-Dataset.csv


## Identify data types

In [2]:
import pandas as pd

df = pd.read_csv(csv_path)

print('Shape:', df.shape)
print()
print('First 5 rows:')
display(df.head())
print()
print('Data types:')
display(df.dtypes)
print()
print('Info:')
df.info()

Shape: (891, 12)

First 5 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



Data types:


PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


Info:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


### Data type summary

| Column | dtype | Description |
|--------|-------|-------------|
| PassengerId | int64 | Unique passenger identifier |
| Survived | int64 | Survival flag (0 = No, 1 = Yes) |
| Pclass | int64 | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd) |
| Name | object | Passenger name (string) |
| Sex | object | Gender (string: male/female) |
| Age | float64 | Age in years |
| SibSp | int64 | # siblings/spouses aboard |
| Parch | int64 | # parents/children aboard |
| Ticket | object | Ticket number (string) |
| Fare | float64 | Passenger fare |
| Cabin | object | Cabin number (string, many NaN) |
| Embarked | object | Port of embarkation (C/Q/S) |

## Transform `Sex` into a two-column (0/1) matrix

In [3]:
# One-hot encode the 'Sex' column
sex_dummies = pd.get_dummies(df['Sex'], prefix='Sex').astype(int)

print('Two-column Sex matrix (first 10 rows):')
display(sex_dummies.head(10))

# Add the encoded columns to the dataframe
df = pd.concat([df, sex_dummies], axis=1)

print('\nDataframe with Sex_female and Sex_male columns:')
display(df[['PassengerId', 'Sex', 'Sex_female', 'Sex_male']].head(10))

Two-column Sex matrix (first 10 rows):


,Sex_female,Sex_male
0,0,1
1,1,0
2,1,0
3,1,0
4,0,1
5,0,1
6,0,1
7,0,1
8,1,0
9,1,0



Dataframe with Sex_female and Sex_male columns:


,PassengerId,Sex,Sex_female,Sex_male
0,1,male,0,1
1,2,female,1,0
2,3,female,1,0
3,4,female,1,0
4,5,male,0,1
5,6,male,0,1
6,7,male,0,1
7,8,male,0,1
8,9,female,1,0
9,10,female,1,0


## Create subset and count selected passengers

**Criteria:**
- Passengers who **survived** AND
- Female passengers **older than 45** OR male passengers **younger than 20**

In [4]:
survived = df['Survived'] == 1
female_over_45 = (df['Sex'] == 'female') & (df['Age'] > 45)
male_under_20  = (df['Sex'] == 'male')   & (df['Age'] < 20)

subset = df[survived & (female_over_45 | male_under_20)].copy()

print('Subset (survived AND (female > 45 OR male < 20)):')
display(subset[['PassengerId', 'Survived', 'Sex', 'Age', 'Sex_female', 'Sex_male']])

Subset (survived AND (female > 45 OR male < 20)):


,PassengerId,Survived,Sex,Age,Sex_female,Sex_male
11,12,1,female,58.00,1,0
15,16,1,female,55.00,1,0
52,53,1,female,49.00,1,0
78,79,1,male,0.83,0,1
125,126,1,male,12.00,0,1
165,166,1,male,9.00,0,1
183,184,1,male,1.00,0,1
193,194,1,male,3.00,0,1
195,196,1,female,58.00,1,0
204,205,1,male,18.00,0,1


## How many passengers were selected?

In [5]:
count = len(subset)
print(f'Number of passengers selected: {count}')

Number of passengers selected: 52


## System information

In [6]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1044-azure
Datetime: 2026-03-05 10:10:05
Python Version: 3.11.14
-----------------------------------
